Paths and reusable definitions were refactored. Stored outputs below are historical, not a fresh execution. See `docs/WORKFLOW.md` at the repository root before rerunning; several stages overwrite results or append Optuna trials.


In [ ]:
# Locate this checkout from the repository root or any notebook subdirectory.
from pathlib import Path
import sys

_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
              if (p / "src" / "steam_recommender" / "paths.py").is_file()), None)
if _root is None:
    raise RuntimeError("Start Jupyter inside the repository checkout.")
sys.path.insert(0, str(_root / "src"))
from steam_recommender import paths as _paths
if Path(_paths.__file__).resolve() != (_root / "src/steam_recommender/paths.py").resolve():
    raise RuntimeError("A different checkout is cached; restart the kernel.")
from steam_recommender.paths import project_root, project_path


In [1]:
# =========================
# STATISTICAL TESTS — PAIRED WILCOXON
# =========================

import os
import itertools
import numpy as np
import pandas as pd

from scipy.stats import wilcoxon


# =========================
# PATHS
# =========================


final_results_dir = os.path.join(str(project_path("results/final")))

per_user_ndcg_path = os.path.join(
    final_results_dir,
    "per_user_ndcg.csv"
)

wilcoxon_results_path = os.path.join(
    final_results_dir,
    "wilcoxon_holm_results.csv"
)

wilcoxon_matrix_path = os.path.join(
    final_results_dir,
    "wilcoxon_holm_pvalue_matrix.csv"
)


# =========================
# LOAD DATA
# =========================

per_user_df = pd.read_csv(per_user_ndcg_path)

display(per_user_df.head())
print("per_user_df:", per_user_df.shape)

print(per_user_df["model"].value_counts())

,model,function,function_label,seed,user_id,ndcg@10
0,ALS-linear,linear,liniowa,0.0,--000--,0.000000
1,ALS-linear,linear,liniowa,0.0,--ace--,0.098039
2,ALS-linear,linear,liniowa,0.0,--ionex--,0.000000
3,ALS-linear,linear,liniowa,0.0,-2SV-vuLB-Kg,0.000000
4,ALS-linear,linear,liniowa,0.0,-404PageNotFound-,0.630043


per_user_df: (1321824, 6)
model
ALS-linear             314720
ALS-log                314720
ALS-log-saturation     314720
ALS-log-inside-beta    314720
Popular                 62944
Name: count, dtype: int64


In [2]:
# =========================
# FILTER ALS MODELS
# =========================

als_functions = [
    "linear",
    "log",
    "log_saturation",
    "log_inside_beta",
]

als_df = per_user_df[
    per_user_df["function"].isin(als_functions)
].copy()

# Upewniamy się, że user_id jest stringiem
als_df["user_id"] = als_df["user_id"].astype(str)

ndcg_col = "ndcg@10"


# =========================
# AVERAGE PER-USER NDCG ACROSS SEEDS
# =========================
# To jest ważne:
# seed nie jest niezależną obserwacją użytkownika.
# Dlatego najpierw liczymy średnie NDCG użytkownika dla danego modelu po seedach.

per_user_mean = (
    als_df
    .groupby(["function", "function_label", "user_id"], as_index=False)[ndcg_col]
    .mean()
)

display(per_user_mean.head())
print("per_user_mean:", per_user_mean.shape)


# =========================
# PIVOT: rows = users, columns = models
# =========================

user_model_matrix = per_user_mean.pivot(
    index="user_id",
    columns="function",
    values=ndcg_col
)

# Zostawiamy tylko użytkowników obecnych we wszystkich modelach
user_model_matrix = user_model_matrix.dropna(subset=als_functions)

display(user_model_matrix.head())
print("user_model_matrix:", user_model_matrix.shape)

,function,function_label,user_id,ndcg@10
0,linear,liniowa,--000--,0.078531
1,linear,liniowa,--ace--,0.146435
2,linear,liniowa,--ionex--,0.000000
3,linear,liniowa,-2SV-vuLB-Kg,0.067451
4,linear,liniowa,-404PageNotFound-,0.629625


per_user_mean: (251776, 4)


function,linear,log,log_inside_beta,log_saturation
user_id,,,,
--000--,0.078531,0.453904,0.401994,0.453904
--ace--,0.146435,0.000000,0.083225,0.000000
--ionex--,0.000000,0.307406,0.313544,0.309553
-2SV-vuLB-Kg,0.067451,0.000000,0.000000,0.000000
-404PageNotFound-,0.629625,0.305235,0.227026,0.305235


user_model_matrix: (62944, 4)


In [3]:
# =========================
# HOLM-BONFERRONI CORRECTION
# =========================

from steam_recommender.statistics import holm_bonferroni


In [4]:
# =========================
# PAIRED WILCOXON TESTS
# =========================

model_pairs = list(itertools.combinations(als_functions, 2))

rows = []

for model_a, model_b in model_pairs:
    a = user_model_matrix[model_a]
    b = user_model_matrix[model_b]

    diff = b - a  # dodatnie = model_b lepszy od model_a

    # Jeśli wszystkie różnice są zerowe, scipy może zwrócić błąd.
    if np.allclose(diff, 0):
        statistic = np.nan
        p_value = 1.0
    else:
        test_result = wilcoxon(
            b,
            a,
            alternative="two-sided",
            zero_method="wilcox",
            correction=False,
            mode="auto",
        )

        statistic = test_result.statistic
        p_value = test_result.pvalue

    rows.append({
        "model_a": model_a,
        "model_b": model_b,
        "comparison": f"{model_a} vs {model_b}",
        "n_users": len(diff),
        "wilcoxon_statistic": statistic,
        "p_value_raw": p_value,

        # effect size jako mediana różnicy per-user NDCG
        # dodatnia wartość oznacza przewagę model_b nad model_a
        "median_difference_b_minus_a": float(np.median(diff)),
        "mean_difference_b_minus_a": float(np.mean(diff)),

        "model_a_mean_ndcg": float(a.mean()),
        "model_b_mean_ndcg": float(b.mean()),
    })

wilcoxon_results = pd.DataFrame(rows)

# Holm-Bonferroni
wilcoxon_results["p_value_holm"] = holm_bonferroni(
    wilcoxon_results["p_value_raw"].values
)

wilcoxon_results["significant_0.05"] = (
    wilcoxon_results["p_value_holm"] < 0.05
)

wilcoxon_results = wilcoxon_results.sort_values(
    "p_value_raw"
).reset_index(drop=True)

display(wilcoxon_results)

wilcoxon_results.to_csv(wilcoxon_results_path, index=False)

print(f"Saved Wilcoxon results to: {wilcoxon_results_path}")

,model_a,model_b,comparison,n_users,wilcoxon_statistic,p_value_raw,median_difference_b_minus_a,mean_difference_b_minus_a,model_a_mean_ndcg,model_b_mean_ndcg,p_value_holm,significant_0.05
0,log_saturation,log_inside_beta,log_saturation vs log_inside_beta,62944,550733889.0,2.243051e-140,0.0,0.016272,0.335418,0.351690,1.345830e-139,True
1,log,log_inside_beta,log vs log_inside_beta,62944,552419387.0,2.330615e-135,0.0,0.016077,0.335613,0.351690,1.165307e-134,True
2,log,log_saturation,log vs log_saturation,62944,99805795.5,1.401539e-16,0.0,-0.000195,0.335613,0.335418,5.606157e-16,True
3,linear,log_inside_beta,linear vs log_inside_beta,62944,677214139.5,1.556183e-13,0.0,0.021760,0.329930,0.351690,4.668550e-13,True
4,linear,log_saturation,linear vs log_saturation,62944,674322535.0,6.771269e-04,0.0,0.005488,0.329930,0.335418,1.354254e-03,True
5,linear,log,linear vs log,62944,674358249.0,1.477462e-03,0.0,0.005683,0.329930,0.335613,1.477462e-03,True


Saved Wilcoxon results to: D:\Studiowe\Licencjat\modele\final_results\wilcoxon_holm_results.csv


In [6]:
# =========================
# ADJUSTED P-VALUE MATRIX
# =========================

pval_matrix = pd.DataFrame(
    np.nan,
    index=als_functions,
    columns=als_functions
)

for _, row in wilcoxon_results.iterrows():
    a = row["model_a"]
    b = row["model_b"]
    p = row["p_value_holm"]

    pval_matrix.loc[a, b] = p
    pval_matrix.loc[b, a] = p

for i in range(len(pval_matrix)):
    pval_matrix.iloc[i, i] = 0.0

display(pval_matrix)

pval_matrix.to_csv(wilcoxon_matrix_path)

print(f"Saved p-value matrix to: {wilcoxon_matrix_path}")

,linear,log,log_saturation,log_inside_beta
linear,0.000000e+00,1.477462e-03,1.354254e-03,4.668550e-13
log,1.477462e-03,0.000000e+00,5.606157e-16,1.165307e-134
log_saturation,1.354254e-03,5.606157e-16,0.000000e+00,1.345830e-139
log_inside_beta,4.668550e-13,1.165307e-134,1.345830e-139,0.000000e+00


Saved p-value matrix to: D:\Studiowe\Licencjat\modele\final_results\wilcoxon_holm_pvalue_matrix.csv


In [7]:
# =========================
# PRETTY TABLE
# =========================

function_labels = {
    "linear": "liniowa",
    "log": "logarytmiczna",
    "log_saturation": "logarytmiczna z saturacją",
    "log_inside_beta": "logarytmiczna z przesunięciem β",
}

pretty_wilcoxon = wilcoxon_results.copy()

pretty_wilcoxon["Porównanie"] = (
    pretty_wilcoxon["model_a"].map(function_labels)
    + " vs "
    + pretty_wilcoxon["model_b"].map(function_labels)
)

pretty_wilcoxon["Mediana różnicy NDCG@10"] = (
    pretty_wilcoxon["median_difference_b_minus_a"]
)

pretty_wilcoxon["p-value"] = pretty_wilcoxon["p_value_raw"]
pretty_wilcoxon["p-value Holm"] = pretty_wilcoxon["p_value_holm"]
pretty_wilcoxon["Istotne α=0.05"] = pretty_wilcoxon["significant_0.05"]

pretty_wilcoxon = pretty_wilcoxon[
    [
        "Porównanie",
        "n_users",
        "Mediana różnicy NDCG@10",
        "p-value",
        "p-value Holm",
        "Istotne α=0.05",
    ]
]

display(pretty_wilcoxon)

,Porównanie,n_users,Mediana różnicy NDCG@10,p-value,p-value Holm,Istotne α=0.05
0,logarytmiczna z saturacją vs logarytmiczna z p...,62944,0.0,2.243051e-140,1.345830e-139,True
1,logarytmiczna vs logarytmiczna z przesunięciem β,62944,0.0,2.330615e-135,1.165307e-134,True
2,logarytmiczna vs logarytmiczna z saturacją,62944,0.0,1.401539e-16,5.606157e-16,True
3,liniowa vs logarytmiczna z przesunięciem β,62944,0.0,1.556183e-13,4.668550e-13,True
4,liniowa vs logarytmiczna z saturacją,62944,0.0,6.771269e-04,1.354254e-03,True
5,liniowa vs logarytmiczna,62944,0.0,1.477462e-03,1.477462e-03,True
